# BuzzSpot Pollinator Detection — Baseline (clean)

End-to-end: locate dataset, build YOLO dataset, train YOLO11s for 8 epochs, validate, generate submission.
Run cells top to bottom.

## 1. Install

In [ ]:
!pip install -q ultralytics pycocotools tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 19.4 MB/s eta 0:00:00


## 2. Mount Drive and locate dataset zip

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import glob, os
hits = glob.glob("/content/drive/MyDrive/**/BuzzSet_challenge.zip", recursive=True)
assert hits, "BuzzSet_challenge.zip not found under MyDrive"
DRIVE_ZIP = hits[0]
print("zip:", DRIVE_ZIP)

Mounted at /content/drive
zip: /content/drive/MyDrive/Colab Notebooks/CVPPA ECCV 26 BuzzSpot Challenge/BuzzSet_challenge.zip


## 3. Build YOLO dataset (annotated keyframes only)

In [ ]:
import zipfile, json
from pathlib import Path
from collections import defaultdict

OUT = Path("/content/buzz")
for d in ["images/train","images/val","images/test","labels/train","labels/val"]:
    (OUT/d).mkdir(parents=True, exist_ok=True)

zf = zipfile.ZipFile(DRIVE_ZIP)
names = set(zf.namelist())

def zpath(rel):
    for cand in (rel, f"BuzzSet_challenge/{rel}"):
        if cand in names:
            return cand
    return None

def load_ann(fname):
    p = zpath(f"annotations/{fname}")
    assert p is not None, f"{fname} not in zip"
    with zf.open(p) as f:
        return json.load(f)

train = load_ann("train.json")
valid = load_ann("valid.json")
test  = load_ann("test_devphase.json")

def flat(fn): return fn.replace("/","__")

def extract(coco, split, img_subdir, write_labels):
    by_img = defaultdict(list)
    for a in coco.get("annotations", []):
        by_img[a["image_id"]].append(a)
    images = {im["id"]: im for im in coco["images"]}
    ids = list(by_img.keys()) if write_labels else [im["id"] for im in coco["images"]]
    id_to_flat = {}
    for iid in ids:
        im = images[iid]; fn = im["file_name"]
        src = zpath(f"{img_subdir}/{fn}")
        if src is None: continue
        out_name = flat(fn)
        with zf.open(src) as s, open(OUT/"images"/split/out_name,"wb") as d:
            d.write(s.read())
        id_to_flat[iid] = out_name
        if write_labels:
            W,H = im["width"], im["height"]; lines=[]
            for a in by_img[iid]:
                x,y,w,h = a["bbox"]
                lines.append(f"{a['category_id']-1} {(x+w/2)/W:.6f} {(y+h/2)/H:.6f} {w/W:.6f} {h/H:.6f}")
            (OUT/"labels"/split/(out_name.rsplit('.',1)[0]+".txt")).write_text("\n".join(lines))
    return id_to_flat

print("extracting train..."); extract(train,"train","train",True)
print("extracting val...");   extract(valid,"val","valid",True)
print("extracting test...");  test_id_to_flat = extract(test,"test","test_devphase",False)
json.dump(test_id_to_flat, open("/content/test_id_to_flat.json","w"))

n_train=len(list((OUT/"images/train").glob("*")))
n_val=len(list((OUT/"images/val").glob("*")))
n_test=len(list((OUT/"images/test").glob("*")))
print(f"train imgs: {n_train} | val imgs: {n_val} | test imgs: {n_test}")
assert n_train>0 and n_val>0 and n_test==len(test["images"]), "extraction problem"

import yaml
yaml.safe_dump({"path":"/content/buzz","train":"images/train","val":"images/val",
                "names":{0:"bee",1:"bumblebee",2:"hoverfly",3:"moth"}},
               open("/content/buzz/data.yaml","w"))
print("data.yaml written")

extracting train...
extracting val...
extracting test...
train imgs: 5275 | val imgs: 932 | test imgs: 12888
data.yaml written


## 4. Train YOLO11s — 8 epochs, fresh

Trains on the BuzzSet dataset built above. Writes to `/content/runs/buzz`. If the T4 OOMs at batch=8 it auto-reduces to 4.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")
model.train(
    data="/content/buzz/data.yaml",
    epochs=8,
    imgsz=1280,
    batch=8,
    project="/content/runs",
    name="buzz",
    exist_ok=True,
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.78 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/buzz/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=8, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7c47b1aca180>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pathlib import Path
import time

matches = []

for p in Path("/content/drive/MyDrive").rglob("best.pt"):
    try:
        size = p.stat().st_size
        mtime = p.stat().st_mtime
        matches.append((mtime, size, p))
    except:
        pass

matches = sorted(matches, reverse=True)

for i, (mtime, size, p) in enumerate(matches[:20]):
    print(i, round(size / 1e6, 2), "MB", time.ctime(mtime), p)

0 19.24 MB Thu Jun 25 20:44:24 2026 /content/drive/MyDrive/Colab Notebooks/CVPPA ECCV 26 BuzzSpot Challenge/runs/baseline/weights/best.pt


In [ ]:
import shutil

BEST_DRIVE = matches[0][2]
print("using:", BEST_DRIVE)

shutil.copy(BEST_DRIVE, "/content/best.pt")
print("copied to /content/best.pt")

using: /content/drive/MyDrive/Colab Notebooks/CVPPA ECCV 26 BuzzSpot Challenge/runs/baseline/weights/best.pt
copied to /content/best.pt


## 5. Validate (per-class)

In [ ]:
from pathlib import Path
from ultralytics import YOLO

BEST = Path("/content/best.pt")
assert BEST.exists(), "/content/best.pt is missing"

model = YOLO(str(BEST))
print("classes:", model.names)

classes: {0: 'bee', 1: 'bumblebee', 2: 'hoverfly', 3: 'moth'}


## 6. Inference on test set → predictions.json → submission.zip

In [ ]:
import json, glob
import zipfile
from pathlib import Path
from ultralytics import YOLO

model = YOLO("/content/best.pt")

# Find dataset zip in Drive
zip_matches = glob.glob("/content/drive/MyDrive/**/BuzzSet_challenge.zip", recursive=True)
assert zip_matches, "Could not find BuzzSet_challenge.zip in Drive"
DRIVE_ZIP = zip_matches[0]

# Read test_devphase.json from the zip
with zipfile.ZipFile(DRIVE_ZIP) as zf:
    json_files = [
        n for n in zf.namelist()
        if n.endswith("test_devphase.json") and "/._" not in n and not Path(n).name.startswith("._")
    ]
    assert json_files, "Could not find test_devphase.json in zip"
    test_json_name = json_files[0]
    test = json.load(zf.open(test_json_name))

def flat_name(file_name):
    return file_name.replace("/", "__")

# Exact keyframe selection
keyframe_images = [
    im for im in test["images"]
    if im.get("is_keyframe") in [True, 1, "true", "True"]
]

flat_to_id = {
    flat_name(im["file_name"]): im["id"]
    for im in keyframe_images
}

TEST_DIR = Path("/content/buzz/images/test")

keyframe_paths = [
    str(TEST_DIR / flat_name(im["file_name"]))
    for im in keyframe_images
]

missing = [p for p in keyframe_paths if not Path(p).exists()]

print("total test images in json:", len(test["images"]))
print("keyframes selected:", len(keyframe_paths))
print("missing keyframe files:", len(missing))
print("sample keyframes:", keyframe_paths[:5])

assert len(keyframe_paths) == 2148, "Unexpected keyframe count"
assert len(missing) == 0, "Some keyframe image files are missing"

total test images in json: 12888
keyframes selected: 2148
missing keyframe files: 0
sample keyframes: ['/content/buzz/images/test/test_video_1__test_video_1_005730.jpg', '/content/buzz/images/test/test_video_1__test_video_1_007500.jpg', '/content/buzz/images/test/test_video_1__test_video_1_007950.jpg', '/content/buzz/images/test/test_video_1__test_video_1_017730.jpg', '/content/buzz/images/test/test_video_1__test_video_1_024720.jpg']


In [ ]:
import json, zipfile, gc, torch
from pathlib import Path

# clear previous failed generator/results
try:
    del results
except:
    pass

gc.collect()
torch.cuda.empty_cache()

CONF = 0.01
W, H = 1920, 1080
preds = []

for i, img_path in enumerate(keyframe_paths):
    with torch.inference_mode():
        r = model.predict(
            source=str(img_path),
            imgsz=1280,
            conf=CONF,
            iou=0.6,
            max_det=100,
            batch=1,
            verbose=False
        )[0]

    fname = Path(r.path).name
    iid = flat_to_id.get(fname)

    if iid is not None:
        boxes = r.boxes

        for b in boxes:
            x1, y1, x2, y2 = b.xyxy[0].tolist()

            x1 = max(0.0, min(x1, W - 1))
            y1 = max(0.0, min(y1, H - 1))
            x2 = max(0.0, min(x2, W))
            y2 = max(0.0, min(y2, H))

            w = x2 - x1
            h = y2 - y1

            if w >= 1 and h >= 1:
                preds.append({
                    "image_id": int(iid),
                    "category_id": int(b.cls[0]) + 1,
                    "bbox": [round(x1, 2), round(y1, 2), round(w, 2), round(h, 2)],
                    "score": round(float(b.conf[0]), 5)
                })

    del r

    if i % 100 == 0:
        print(i, "/", len(keyframe_paths), "preds:", len(preds))
        gc.collect()
        torch.cuda.empty_cache()

with open("/content/predictions.json", "w") as f:
    json.dump(preds, f)

with zipfile.ZipFile("/content/submission_keyframes_conf001.zip", "w") as z:
    z.write("/content/predictions.json", "predictions.json")

print("detections:", len(preds))
print("images with predictions:", len({p["image_id"] for p in preds}))
print("wrote /content/submission_keyframes_conf001.zip")

0 / 2148 preds: 12
100 / 2148 preds: 649
200 / 2148 preds: 1336
300 / 2148 preds: 1933
400 / 2148 preds: 2437
500 / 2148 preds: 3035
600 / 2148 preds: 3452
700 / 2148 preds: 3822
800 / 2148 preds: 4336
900 / 2148 preds: 4858
1000 / 2148 preds: 5346
1100 / 2148 preds: 5921
1200 / 2148 preds: 6381
1300 / 2148 preds: 6869
1400 / 2148 preds: 7175
1500 / 2148 preds: 7442
1600 / 2148 preds: 7836
1700 / 2148 preds: 8226
1800 / 2148 preds: 8566
1900 / 2148 preds: 8735
2000 / 2148 preds: 9095
2100 / 2148 preds: 9488
detections: 9973
images with predictions: 2141
wrote /content/submission_keyframes_conf001.zip


## 7. Validate submission before upload

In [ ]:
import json, zipfile
from pathlib import Path

p = json.load(open("/content/predictions.json"))
ids = {d["image_id"] for d in p}

# keyframe_images should exist from the previous keyframe-selection cell
keyframe_ids = {im["id"] for im in keyframe_images}

print("detections:", len(p))
print("distinct predicted images:", len(ids))
print("total keyframe images:", len(keyframe_ids))
print("predictions outside keyframes:", len(ids - keyframe_ids))
print("keyframes with no detections:", len(keyframe_ids - ids))
print("categories:", sorted({d["category_id"] for d in p}))

print("degenerate:", sum(1 for d in p if d["bbox"][2] < 1 or d["bbox"][3] < 1))
print("out of bounds:", sum(
    1 for d in p
    if d["bbox"][0] < -0.5
    or d["bbox"][1] < -0.5
    or d["bbox"][0] + d["bbox"][2] > 1920.5
    or d["bbox"][1] + d["bbox"][3] > 1080.5
))

with zipfile.ZipFile("/content/submission_keyframes_conf001.zip") as z:
    print("zip contents:", z.namelist())

print("\nUpload /content/submission_keyframes_conf001.zip")

detections: 9973
distinct predicted images: 2141
total keyframe images: 2148
predictions outside keyframes: 0
keyframes with no detections: 7
categories: [1, 2, 3, 4]
degenerate: 0
out of bounds: 0
zip contents: ['predictions.json']

Upload /content/submission_keyframes_conf001.zip
